In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


# Silver to Gold Transform
## Purpose
Splits silver_orders into a star schema Gold layer.
## Output tables
- dim_customer (customer_key, customer_id, customer_name, segment)
- dim_product (product_key, product_id, product_name, category, sub_category)
- dim_date (date_key, order_date, year, quarter, month, month_name,
day_of_week)
- fact_orders (order_id, customer_key, product_key, date_key, region, sales,
quantity, discount, profit)


In [3]:
df = spark.table('silver_orders')
print(f'Silver rows: {df.count()}')
df.printSchema()

StatementMeta(, 11289f7f-209b-4ed5-be24-a448abc7e25b, 12, Finished, Available, Finished, False)

Silver rows: 10194
root
 |-- Row_ID: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- State/Province: string (nullable = true)
 |-- Postal_Code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- sales: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [3]:
from pyspark.sql.functions import monotonically_increasing_id
dim_customer = df.select('customer_id','customer_name','segment') \
.distinct() \
.withColumn('customer_key', monotonically_increasing_id())
# Reorder columns: key first
dim_customer = dim_customer.select('customer_key','customer_id','customer_name','segment')
dim_customer.write.mode('overwrite').format('delta').saveAsTable('dim_customer')
print(f'dim_customer rows: {dim_customer.count()}')
dim_customer.show(5)

StatementMeta(, c32b02ad-2093-4596-8b9d-1ed528953f49, 11, Finished, Available, Finished, False)

dim_customer rows: 804
+------------+-----------+--------------+-----------+
|customer_key|customer_id| customer_name|    segment|
+------------+-----------+--------------+-----------+
|           0|   LS-17200|  Luke Schmidt|  Corporate|
|           1|   PN-18775|Parhena Norris|Home Office|
|           2|   DP-13105|  Dave Poirier|  Corporate|
|           3|   GH-14665|   Greg Hansen|   Consumer|
|           4|   MG-17875| Michael Grace|Home Office|
+------------+-----------+--------------+-----------+
only showing top 5 rows



In [5]:
dim_product = df.select('product_id','product_name','category','sub_category') \
.distinct() \
.withColumn('product_key', monotonically_increasing_id())
dim_product = dim_product.select('product_key','product_id','product_name','category','sub_category'
)
dim_product.write.mode('overwrite').format('delta').saveAsTable('dim_product')
print(f'dim_product rows: {dim_product.count()}')
dim_product.show(5)

StatementMeta(, c32b02ad-2093-4596-8b9d-1ed528953f49, 18, Finished, Available, Finished, False)

dim_product rows: 1894
+-----------+---------------+--------------------+---------------+------------+
|product_key|     product_id|        product_name|       category|sub_category|
+-----------+---------------+--------------------+---------------+------------+
|          0|OFF-PA-10004948|           Xerox 190|Office Supplies|       Paper|
|          1|TEC-PH-10002033|Konftel 250 Confe...|     Technology|      Phones|
|          2|OFF-BI-10000069|GBC Prepunched Pa...|Office Supplies|     Binders|
|          3|OFF-PA-10003971|          Xerox 1965|Office Supplies|       Paper|
|          4|FUR-CH-10002335|Hon GuestStacker ...|      Furniture|      Chairs|
+-----------+---------------+--------------------+---------------+------------+
only showing top 5 rows



In [10]:
from pyspark.sql.functions import (
year, quarter, month, dayofweek,
date_format, col, to_date
)
from pyspark.sql.types import IntegerType
# Extract all unique order dates from silver_orders
dim_date = df.select('order_date').distinct()
# Add calendar attributes
dim_date = dim_date \
.withColumn('year', year(col('order_date')).cast(IntegerType())) \
.withColumn('quarter', quarter(col('order_date')).cast(IntegerType())) \
.withColumn('month', month(col('order_date')).cast(IntegerType())) \
.withColumn('month_name', date_format(col('order_date'), 'MMMM')) \
.withColumn('day_of_week', date_format(col('order_date'), 'EEEE')) \
.withColumn('date_key', date_format(col('order_date'),'yyyyMMdd').cast(IntegerType()))
# Reorder columns
dim_date = dim_date.select(
'date_key','order_date','year','quarter','month','month_name','day_of_week'
)
dim_date.write.mode('overwrite').format('delta').saveAsTable('dim_date')
print(f'dim_date rows: {dim_date.count()}')
dim_date.show(5)

StatementMeta(, c32b02ad-2093-4596-8b9d-1ed528953f49, 33, Finished, Available, Finished, False)

dim_date rows: 1242
+--------+----------+----+-------+-----+----------+-----------+
|date_key|order_date|year|quarter|month|month_name|day_of_week|
+--------+----------+----+-------+-----+----------+-----------+
|20230622|2023-06-22|2023|      2|    6|      June|   Thursday|
|20230715|2023-07-15|2023|      3|    7|      July|   Saturday|
|20240918|2024-09-18|2024|      3|    9| September|  Wednesday|
|20250216|2025-02-16|2025|      1|    2|  February|     Sunday|
|20251021|2025-10-21|2025|      4|   10|   October|    Tuesday|
+--------+----------+----+-------+-----+----------+-----------+
only showing top 5 rows



In [11]:
fact_orders = df \
.join(dim_customer.select('customer_id','customer_key'), on='customer_id',
how='left') \
.join(dim_product.select('product_id','product_key'), on='product_id',
how='left') \
.join(dim_date.select('order_date','date_key'), on='order_date',
how='left') \
.select(
    col('order_id'),
col('customer_key'),
col('product_key'),
col('date_key'),
col('region'),
col('sales'),
col('quantity'),
col('discount'),
col('profit')
)
fact_orders.write.mode('overwrite').format('delta').saveAsTable('fact_orders')
print(f'fact_orders rows: {fact_orders.count()}')
fact_orders.show(5)

StatementMeta(, c32b02ad-2093-4596-8b9d-1ed528953f49, 36, Finished, Available, Finished, False)

fact_orders rows: 10537
+--------------+------------+-----------+--------+-------+-------+--------+--------+--------+
|      order_id|customer_key|product_key|date_key| region|  sales|quantity|discount|  profit|
+--------------+------------+-----------+--------+-------+-------+--------+--------+--------+
|US-2023-103800|         443|        133|20230103|Central|   NULL|      16|     2.0|     0.2|
|US-2023-112326|          43|        566|20230104|Central|   3.54|       2|     0.8|  -5.487|
|US-2023-112326|          43|         25|20230104|Central| 11.784|       3|     0.2|  4.2717|
|US-2023-112326|          43|        507|20230104|Central|272.736|       3|     0.2|-64.7748|
|US-2023-141817|         651|       1713|20230105|   East| 19.536|       3|     0.2|   4.884|
+--------------+------------+-----------+--------+-------+-------+--------+--------+--------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import count, when, col
fact_orders.select([
count(when(col('customer_key').isNull(), 1)).alias('null_customer_key'),
count(when(col('product_key').isNull(), 1)).alias('null_product_key'),
count(when(col('date_key').isNull(), 1)).alias('null_date_key'),
]).show()

StatementMeta(, c32b02ad-2093-4596-8b9d-1ed528953f49, 38, Finished, Available, Finished, False)

+-----------------+----------------+-------------+
|null_customer_key|null_product_key|null_date_key|
+-----------------+----------------+-------------+
|                0|               0|            0|
+-----------------+----------------+-------------+



In [1]:
%%sql
select * from dim_customer;

StatementMeta(, b6874317-39da-4806-9fa0-4d7b1c2ff177, 2, Finished, Available, Finished, False)

<Spark SQL result set with 804 rows and 4 fields>

In [2]:
%%sql
select*from dim_date;

StatementMeta(, b6874317-39da-4806-9fa0-4d7b1c2ff177, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 7 fields>

In [3]:
%%sql
select * from dim_product;

StatementMeta(, b6874317-39da-4806-9fa0-4d7b1c2ff177, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 5 fields>

In [4]:
%%sql
select * from fact_orders;

StatementMeta(, b6874317-39da-4806-9fa0-4d7b1c2ff177, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 9 fields>

In [8]:
from pyspark.sql.functions import monotonically_increasing_id

# Create dim_region table
dim_region = df.select('region') \
    .distinct() \
    .withColumn('region_key', monotonically_increasing_id())

# Reorder columns: key first
dim_region = dim_region.select(
    'region_key',
    'region'
)



# Save as Delta table
dim_region.write \
    .mode('overwrite') \
    .format('delta') \
    .saveAsTable('dim_region')

# Check row count
print(f'dim_region rows: {dim_region.count()}')

# Display sample records
dim_region.show(5)

StatementMeta(, 11289f7f-209b-4ed5-be24-a448abc7e25b, 26, Finished, Available, Finished, False)

dim_region rows: 4
+----------+-------+
|region_key| region|
+----------+-------+
|         0|  South|
|         1|Central|
|         2|   East|
|         3|   West|
+----------+-------+

